# 4.2.1-Analysis of SPOTlight results

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
options(warn = -1, verbose = FALSE)
suppressMessages(suppressWarnings({
library(ggplot2)
library(SPOTlight)
library(Seurat)
library(dplyr)
library(NMF)
}))

## Parameters

In [ ]:
args <- commandArgs(TRUE)

pat <- args[2]
subproj <- args[3]
level <- args[4]

In [ ]:
#here::dr_here(show_reason = TRUE)

source(here::here(glue::glue("{subproj}/st_analysis/misc/paths.R")))
set.seed(0)

## Load Data

In [ ]:
info_cohort <- readr::read_csv(here::here("utils/info_samples.csv"))
info_donor <- info_cohort %>% 
  dplyr::filter(patient == pat) # referring a variable with same name as column in dplyr

In [ ]:
res <- readRDS(file = here::here(glue::glue("{ann}/{robj_dir}/4.2.0-{pat}_l2_spotlight.rds")))
# Extract NMF model fit
mod <- res$NMF

In [ ]:
names(res)

In [ ]:
topic_matrix <- coef(res$NMF)

In [ ]:
rhab_tumor <- readRDS(here::here(glue::glue("{ann}/{robj_dir}/{pat}.rds")))

In [ ]:
# paired multiome
ref_obj <- readRDS(here::here(glue::glue("{ann}/{robj_dir}/3.3.0-anno_{pat}.rds")))
DefaultAssay(ref_obj) <- "RNA"

m_subproj <- info_donor[info_donor$modality == "multiome", ]$subproject

In [ ]:
if (level == "l1") {
    colors_list <- list("Immune"="#d73027", "Other stromal"="#a6d96a","Tumor"="#4575b4")
} else {
    source(here::here(glue::glue("{m_subproj}/multiome_analysis/utils/style.R")))
    var_name <- paste0("colors_", pat)
    colors_list <- get(var_name)
}

## Deconvolution QC

In [ ]:
plotTopicProfiles(
    x = mod,
    y = ref_obj$clusters,
    facet = FALSE,
    min_prop = 0.01,
    ncol = 1) + 
    theme(aspect.ratio = 1)

Next plot shows whether all cells of the cluster share a similar topic profile.

In [ ]:
plotTopicProfiles(
    x = mod,
    y = ref_obj$clusters,
    facet = TRUE,
    min_prop = 0.01,
    ncol = 4)

Genes the model learned for each topic. The highest the more relevant

In [ ]:
sign <- mod$w
colnames(sign) <- paste0("Topic", seq_len(ncol(sign)))
head(sign)

In [ ]:
as.data.frame(sign)["COL1A1",]

## Visualization of results

Create level 1 by summing proportions per cell group

In [ ]:
if (level == "l1") {
    # level 1 (less resolution) cell type deconvolution
    data_df <- as.data.frame(res$mat)
    col_names <- names(data_df)

    # group column names by cell group
    tumor_idx <- startsWith(col_names, "Tumor") 

    immune_exact <- col_names %in% c("Immune Cells", "T-cells", "Plasma B-cell","Plasma cells")
    immune_pattern <- grepl("Macrophage", col_names)

    immune_idx <- immune_exact | immune_pattern

    used_idx <- tumor_idx | immune_idx
    stromal_idx <- !used_idx

    # sum columns per cell group
    Tumor_Sum <- rowSums(data_df[, tumor_idx, drop = FALSE], na.rm = TRUE)

    Immune_Sum <- rowSums(data_df[, immune_idx, drop = FALSE], na.rm = TRUE)

    Other_Stromal_Sum <- rowSums(data_df[, stromal_idx, drop = FALSE], na.rm = TRUE)

    processed_df <- data.frame(
      Tumor = Tumor_Sum,
      Immune = Immune_Sum,
      `Other stromal` = Other_Stromal_Sum,
      check.names = FALSE 
    )
    mat <- as.matrix(processed_df)
} else {
    # level 2 (more resolution) cell type deconvolution
    mat <- res$mat
}

In [ ]:
pc <- plotCorrelationMatrix(mat)
pc

In [ ]:
ggsave(plot = pc, width = 8, height = 8, dpi = 300, 
       file = here::here(glue::glue("{ann}/{plt_dir}/4.2.1-deconv_corr_{pat}_{level}.svg")))

In [ ]:
plotInteractions(mat, which = "heatmap", metric = "prop")

In [ ]:
pco <- plotInteractions(mat, which = "heatmap", metric = "jaccard")

In [ ]:
ggsave(plot = pco, width = 8, height = 8, dpi = 300, 
       file = here::here(glue::glue("{ann}/{plt_dir}/4.2.1-deconv_colo_{pat}_{level}.svg")))

In [ ]:
plotInteractions(mat, which = "network")

### ScatterPie

In [ ]:
spatial_coords <- rhab_tumor@images[[pat]]@coordinates[, c("imagecol", "imagerow")]
coords <- spatial_coords[rownames(mat), c(2,1)]
colnames(coords) <- c("coord_x", "coord_y")

In [ ]:
head(mat)

In [ ]:
all(rownames(coords) == rownames(mat))

In [ ]:
df <- cbind(coords, as.data.frame(mat))

In [ ]:
dist_mat <- as.matrix(dist(coords))
diag(dist_mat) <- Inf # Ignore distance to self
spot_pitch <- min(dist_mat) 

spot_radius <- (spot_pitch * 0.55 / 2) * 1.5 

df$radius <- spot_radius

In [ ]:
ps <- ggplot() +
        scatterpie::geom_scatterpie(
            data = df,
            aes(x = coord_y, y = coord_x,r = radius),
            cols = colnames(mat),
            alpha = 1,
            pie_scale = 1, 
            color = NA
    ) +
    coord_fixed(ratio = 1, expand = FALSE) +
    scale_y_reverse() + 
    theme(
        panel.background = element_blank(),
        plot.background = element_blank(),
        panel.grid = element_blank(),
        axis.line = element_blank(),
        axis.ticks = element_blank(),
        axis.text = element_blank(),
        axis.title = element_blank()
      ) + 
    scale_fill_manual(
       values = as.vector(colors_list),
       breaks = names(colors_list))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 6)
ps

In [ ]:
ggsave(plot = ps, width = 8, height = 6, dpi = 150, 
        file = here::here(glue::glue("{ann}/{plt_dir}/4.2.1-deconv_{pat}_{level}.svg")))

In [ ]:
ps <- ps + 
    labs(title = level) + # Or "" if you want it blank
    theme_void() +
    theme(
        plot.title = element_text(face="bold", hjust = 0.5),
        legend.position = "right", # Must match the other plots
        legend.title = element_blank(),
        legend.text = element_blank(),
        legend.key = element_blank(),
        legend.key.size = unit(0, "pt") # Make the dummy legend invisible
    ) + coord_fixed(ratio = 1, expand = FALSE)

In [ ]:
decon_df <- mat %>%
  data.frame() 

colnames(decon_df) <- colnames(mat)
decon_df$barcodes <- rownames(decon_df)
readr::write_csv(decon_df[,c("barcodes",colnames(mat))], here::here(glue::glue("{ann}/{robj_dir}/4.2.1-norm_weights_{pat}_{level}.csv")))

rhab_tumor@meta.data <- rhab_tumor@meta.data %>%
  tibble::rownames_to_column("barcodes") %>%
  dplyr::left_join(decon_df, by = "barcodes") %>%
  tibble::column_to_rownames("barcodes")

In [ ]:
head(decon_df)

In [ ]:
library(ggforce)
scaleFUN <- function(x) sprintf("%.2f", x)
# Loop through cell types
ct_all <- colnames(mat)
ct_plt_ls <- lapply(ct_all, function(ct) {
  
  df$value_to_plot <- rhab_tumor@meta.data[[ct]] 
  
  tmp_plt <- ggplot(df) +
    ggforce::geom_circle(
      aes(x0 = coord_y, y0 = coord_x, r = radius, 
          fill = value_to_plot, 
          color = value_to_plot,
          alpha = value_to_plot),
      linewidth = 0.01 # Minimal border
    ) +
    coord_fixed(ratio = 1) +
    scale_y_reverse() +
    labs(title = ct, fill = "Prop") +
    theme_void() +
    scale_alpha_continuous(range = c(0, 1), guide = "none") +
    scale_fill_viridis_c(
      option = "D",        
      direction = -1,        
      breaks = c(0, 0.5, 1),
      limits = c(0, 1),
      labels = scaleFUN
    ) +
    scale_color_viridis_c(
      option = "D",        
      direction = -1,        
      limits = c(0, 1),
      guide = "none"
    )  +
    theme(
      plot.title = element_text(face="bold", hjust = 0.5),
      legend.position = "right"
    ) 
  tmp_plt <- tmp_plt + coord_fixed(ratio = 1, expand = FALSE)
  # Rasterize if needed (only the circles)
  tmp_plt_rasterized <- ggrastr::rasterise(
    tmp_plt, 
    layers = 'Polygon', # geom_circle creates polygons
    dpi = 600
  )
  
  return(tmp_plt_rasterized)
})

In [ ]:
# Combine ps with list
all_plots <- c(list(ps), ct_plt_ls)

ct_grid <- patchwork::wrap_plots(all_plots, ncol = 5) & 
           theme(plot.margin = margin(5, 5, 5, 5)) # Ensure consistent margins

ct_grid <- ct_grid + patchwork::plot_layout(guides = 'collect')

In [ ]:
options(repr.plot.width = 18, repr.plot.height =10, warn=-1,verbose = FALSE)
ggsave(plot = ct_grid, width = 18, height = 10, dpi = 300, 
       file = here::here(glue::glue("{ann}/{plt_dir}/4.2.1-deconv_inv_{pat}_{level}.svg")))

ct_grid

### Residuals

In [ ]:
rhab_tumor$res_ss <- res[[2]][colnames(rhab_tumor)]
SpatialFeaturePlot(rhab_tumor, features="res_ss", alpha = c(0, 1), image.alpha = 0) + labs(title = "Residuals") +theme(plot.title = element_text(family="Arial",  face="bold", hjust = 0.5, vjust=10)) +  theme(legend.position = "right")

## Session Info 

In [ ]:
sessionInfo()